In [1]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [4]:
from mlde_analysis.default_params import *
variable = "pr"
domain = "uk"
frequency = "1hr"
scenario = "rcp85"
collection = "land-cpm"
resolution = "2.2km"

'/home/vf20964/furflex/data/demo/demo-data'

In [3]:
import functools
import math
import string

import IPython
from IPython.display import HTML
import matplotlib
from matplotlib import animation
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import os
import cf_xarray

from mlde_utils import cp_model_rotated_pole, VariableMetadata
from mlde_analysis import DERIVED_DATA, plot_map
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.data import si_to_mmhour

In [5]:
import os


'/home/vf20964/furflex/data/demo'

In [20]:
target_ds = xr.open_dataset(f"{os.getenv("DATA_PATH")}/datasets/engwales_ccpm-4x-cpmgem_12em_future_1hr_pr/val/predictands.zarr/").isel(ensemble_member=0)
predictors_ds = xr.open_dataset(f"{os.getenv("DATA_PATH")}/datasets/engwales_ccpm-4x-cpmgem_12em_future_1hr_pr/val/predictors.zarr/").isel(ensemble_member=0)
pred_ds = xr.concat([
    xr.open_dataset(
        f"{os.getenv("WORKDIRS_PATH")}/no-vae-invertable-sqrt-standardize-12em-pSTV-CPMGEM-daily-future-1hr-pr/CPMLatte-S-8-F24S1-cpm/samples/0060/val/predictions-{uuid}.zarr"
    ).isel(ensemble_member=0) for uuid in ["dR6sbzak3huZcscxvGSFwE", "DCG6S3sNsVjfAadFLdzrLf", "kUAZMoHZXonU4sxNBt4x5d"]
], dim="sample")
pred_ds = pred_ds.assign_coords(grid_latitude=target_ds["grid_latitude"].copy(), grid_longitude=target_ds["grid_longitude"].copy())
# pred_ds["rotated_latitude_longitude"] = ([], 0, {**truth_ds["rotated_latitude_longitude"].attrs})
# pred_ds["pr"] = (("time", "frame", "grid_longitude", "grid_latitude"), np.transpose(pred_ds["pr"].values, (0,1,3,2)))


# pred_ds = pred_ds.transpose("time", "frame", "grid_latitude", "grid_longitude")

In [22]:
pred_ds

<xarray.Dataset> Size: 14MB
Dimensions:                     (sample: 3, time: 12, frame: 24,
                                 grid_latitude: 64, grid_longitude: 64)
Coordinates:
  * time                        (time) object 96B 2065-09-01 00:00:00 ... 207...
  * grid_latitude               (grid_latitude) float64 512B -2.49 ... 2.55
  * grid_longitude              (grid_longitude) float64 512B 357.9 ... 363.0
    ensemble_member             <U12 48B 'r001i1p00000'
Dimensions without coordinates: sample, frame
Data variables:
    pr                          (sample, time, frame, grid_latitude, grid_longitude) float32 14MB ...
    rotated_latitude_longitude  (sample) int64 24B 0 0 0

In [23]:
# ds = xr.open_dataset(
#     VariableMetadata(
#         base_dir=DERIVED_DATA/"moose",
#         variable=variable,
#         domain=domain,
#         frequency=frequency,
#         resolution=resolution,
#         scenario=scenario,
#         collection=collection,
#         ensemble_member="01",
#     ).filepath(1981)
# )#.isel(time=slice(0,24))
# ds["pr"] = si_to_mmhour(ds["pr"])
# ds

In [30]:
# em_time = ("01", cftime.Datetime360Day(1993, 8, 1, 12))

# plot_ds = EVAL_DS["CPM"].sel(time=em_time[1], method="nearest").sel(ensemble_member=em_time[0]).sel(model=list(MODELS["CPM"].keys())[-1])
# plot_ds = ds

subplot_kw = dict(projection=cp_model_rotated_pole)

nsamples = 3
nframes = 24
start_day = 0

thetas = [850, 700, 500, 250]

possible_variables = [("spechum", thetas, "Specific\nHumidity\n(multi-level)"), ("temp", thetas, "Temperature\n(multi-level)"), ("vorticity", thetas, "Vorticity\n(multi-level)"), ("psl", [""], "Sea-level pressure")]
variables = possible_variables # list(filter(lambda v: any([k.startswith(v[0]) for k in plot_ds.variables.keys()]), possible_variables))

full_variable_set = [f"{varclass}{level}" for varclass, levels, _title in variables for level in levels]


awidth = 0.12
offset_width = 0.009
stacked_awidth = (awidth + offset_width *3)
gap = (1 - stacked_awidth * 4)/(len(variables) - 1)
print(stacked_awidth*4 + gap*3)

fig = plt.figure(figsize=(6.5, 5.5), layout="constrained")

axd = fig.subplot_mosaic([full_variable_set + [f"pred_pr {i}" for i in range(nsamples)] + ["AI"]], subplot_kw=subplot_kw)

ax = axd["AI"]
# ax.axis("off")
ax.set_facecolor('black')
ax.text(0.5, 0.5, "CPMGEM",
        ha='center', va='center', color="white", weight='bold', transform=ax.transAxes)
ax.set_position([0.43, awidth+0.15, 0.15, 0.15])

output_arrows = [
    dict(
        xy=(1, 1),
        xytext=(0.5, 0),
        arrowprops=dict(facecolor='black', shrinkB=5, arrowstyle="fancy", connectionstyle="arc3,rad=-0.2"),
    ),
    dict(
        xy=(0.5, 1),
        xytext=(0.5, 0),
        arrowprops=dict(facecolor='black', shrinkB=5, arrowstyle="fancy"),
    ),
    dict(
        xy=(0, 1),
        xytext=(0.5, 0),
        arrowprops=dict(facecolor='black', shrinkB=5, arrowstyle="fancy", connectionstyle="arc3,rad=0.2"),
    ),
]
pr_quads = []
for sampleidx in range(nsamples):
    ax = axd[f"pred_pr {sampleidx}"]
    pr_quad = plot_map(pred_ds["pr"].isel(time=start_day, frame=0, sample=sampleidx), ax=ax, style="pr_hourly")
    pr_quads.append(pr_quad)
    ax.set_position([(0.505-awidth/2)+(sampleidx-1)*(awidth+0.03), awidth/2, awidth, awidth])

    if sampleidx == 1:
        ax.text(0.5, -0.15, "High-resolution precipitation", fontsize="small", ha='center', va='center',transform=ax.transAxes)

    axd["AI"].annotate(
            '',
            xycoords=ax.transAxes,
            textcoords=axd["AI"].transAxes,
            **output_arrows[sampleidx],
        )
arrows = [
    dict(
        xy=(0.5, 0.5),
        xytext=(0.5, 0.5),
        arrowprops=dict(facecolor='black', shrinkA=38, shrinkB=33, arrowstyle="simple"),#, connectionstyle="arc3,rad=0.2"),
    ),
    dict(
        xy=(0.5, 0.5),
        xytext=(0.85, -0.25),
        arrowprops=dict(facecolor='black', shrinkB=24, arrowstyle="simple"),#, connectionstyle="arc3,rad=0.2"),
    ),
    dict(
        xy=(0.5, 0.5),
        xytext=(0.5, -0.25),
        arrowprops=dict(facecolor='black', shrinkB=24, arrowstyle="simple"),#, connectionstyle="arc3,rad=-0.2"),
    ),
    dict(
        xy=(0.5, 0.5),
        xytext=(0.5, 0.5),
        arrowprops=dict(facecolor='black', shrinkA=27, shrinkB=33, arrowstyle="simple"),#, connectionstyle="arc3,rad=-0.2"),
    ),
]

for vi, (varclass, levels, vartitle) in enumerate(variables):
    variable_set = [f"{varclass}{level}" for level in levels]

    for i, var in enumerate(variable_set):
        ax = axd[var]
        var_plot_kwargs = {}
        if varclass in ["vorticity"]:
            var_plot_kwargs = {"center": 0}
        plot_map(predictors_ds[var].isel(time=start_day), ax=ax, style=None, **var_plot_kwargs)
        # plot_map(plot_ds["pr"].isel(time=0), ax=ax, style=None, **var_plot_kwargs)
        left = (stacked_awidth + gap) * vi + offset_width*i
        top = 0.5-offset_width*i
        if vi == 0 or vi == len(variables) - 1:
            top = top - 0.1
        ax.set_position([left, top, awidth, awidth])
        if i == 0:
            ax.set_title(vartitle, fontsize="small")
        if i == 0:#len(variable_set)-1:
            axd["AI"].annotate(
                '',
                xycoords=axd["AI"].transAxes,
                textcoords=ax.transAxes,
                **arrows[vi],
            )

def update(frame):
    updated_pr_quads = []
    for sampleidx in range(nsamples):
        ax = axd[f"pred_pr {sampleidx}"]
        pr_quad = plot_map(pred_ds["pr"].isel(time=start_day // 24, frame=frame % 24, sample=sampleidx), ax=ax, style="pr_hourly")
        updated_pr_quads.append(pr_quad)
    
    return updated_pr_quads

anim = animation.FuncAnimation(fig, update, frames=nframes, blit=False)
plt.close(fig)  # prevent double display in notebook
# Display as HTML5 video (works in JupyterLab)
HTML(anim.to_html5_video())

1.0
